# SVCM — ITI-97 — Expansion ValueSet

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-97
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12208&cid=48550

Deux scénarios :
1. `$expand` POST avec ValueSet implicite SNOMED (structures anatomiques, code `302553009`)
2. `$expand` POST avec ValueSet inline CIM-11 (code `04`)

## Configuration

In [1]:
import requests
import json
import time
import os, getpass
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://gazelle-proxypatans.kereval.cloud:10102/fhir"
#FHIR_BASE = os.environ.get("FHIR_BASE", "https://smt.esante.gouv.fr/fhir")

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_POST = {
    "Accept": "application/fhir+json",
    "Content-Type": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "04_iti97_expand"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def http_post(url, body=None, headers=None):
    if headers is None:
        headers = HEADERS_POST
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → POST {url}")
            r = requests.post(url, json=body, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


Configuration OK — sortie dans : runs/04_iti97_expand_20260311T211449


---
## Étapes 0–30 — Expansion d'un ValueSet SNOMED (`$expand`)

**Requête** : `POST /fhir/ValueSet/$expand`

In [2]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : POST $expand (ValueSet implicite SNOMED, ECL << 91723000)
SNOMED_SYSTEM = "http://snomed.info/sct"
SNOMED_SYSTEM_VERSION_FR = "http://snomed.info/sct/11000315107/version/20250621"
SNOMED_SYSTEM_VERSION_EN = "http://snomed.info/sct/900000000000207008/version/20260301"

HEADERS_POST_JSON = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
expand_body_snomed = {
    "resourceType": "Parameters",
    "parameter": [{
        "name": "valueSet",
        "resource": {
            "resourceType": "ValueSet",
            "compose": {
                "include": [{
                    "system": SNOMED_SYSTEM,
                    "filter": [{"property": "constraint", "op": "=", "value": "<< 91723000"}]
                }]
            }
        }
    }]
}

r_expand = http_post(f"{FHIR_BASE}/ValueSet/$expand", expand_body_snomed, HEADERS_POST_JSON)
vs_expanded = r_expand.json()
save_artifact(30, "expand-snomed-91723000.json", vs_expanded)

# Étape 20 — Réception et intégration
expansion = vs_expanded.get("expansion", {})
contains  = expansion.get("contains", [])
total     = expansion.get("total", len(contains))

# Étape 30 — PREUVE
print("[PREUVE étape 30] ValueSet SNOMED expansé (<< 91723000) :")
print_keys(vs_expanded, "resourceType", "status")
print(f"\nExpansion total      : {total}")
print(f"Concepts retournés   : {len(contains)}")
if contains:
    print("\nPremiers concepts :")
    for c in contains[:10]:
        print(f"  {c.get('system')} | {c.get('code')} — {c.get('display')}")
    if len(contains) > 10:
        print(f"  ... ({len(contains) - 10} autres)")

  → POST https://gazelle-proxypatans.kereval.cloud:10102/fhir/ValueSet/$expand
  ✓ runs/04_iti97_expand_20260311T211449/step030_expand-snomed-91723000.json
[PREUVE étape 30] ValueSet SNOMED expansé (<< 91723000) :
{
  "resourceType": "ValueSet"
}

Expansion total      : 37551
Concepts retournés   : 37551

Premiers concepts :
  http://snomed.info/sct | 56244007 — 10 to 19 percent of body surface
  http://snomed.info/sct | 37491003 — 12 nm filaments
  http://snomed.info/sct | 78777002 — 20 to 29 percent of body surface
  http://snomed.info/sct | 12423009 — 30 to 39 percent of body surface
  http://snomed.info/sct | 36849000 — 40 to 49 percent of body surface
  http://snomed.info/sct | 305024009 — 5/6 interchondral joint
  http://snomed.info/sct | 76152003 — 50 to 59 percent of body surface
  http://snomed.info/sct | 305005006 — 6/7 interchondral joint
  http://snomed.info/sct | 91551007 — 60 to 69 percent of body surface
  http://snomed.info/sct | 64700008 — 7 nm filaments
  ... (37541 a

---
## Étapes 40–70 — Expansion d'un ValueSet CIM-11

**Requête** : `POST /fhir/ValueSet/$expand`

In [3]:
# Étape 40 — Construction de la requête
# Étape 50 — TRANSACTION : POST $expand ValueSet inline CIM-11 (code 04)
CIM11_SYSTEM = "https://smt.esante.gouv.fr/terminologie-cim11-mms"
expand_body_cim11 = {
    "resourceType": "Parameters",
    "parameter": [{
        "name": "valueSet",
        "resource": {
            "resourceType": "ValueSet",
            "compose": {
                "include": [{
                    "system": CIM11_SYSTEM,
                    "filter": [{"property": "code", "op": "is-a", "value": "04"}]
                }]
            }
        }
    }]
}

r_cim11 = http_post(f"{FHIR_BASE}/ValueSet/$expand", expand_body_cim11, HEADERS_POST_JSON)
cim11_expanded = r_cim11.json()
save_artifact(70, "expand-cim11-04.json", cim11_expanded)

# Étape 60 — Réception et intégration
expansion_cim11 = cim11_expanded.get("expansion", {})
contains_cim11  = expansion_cim11.get("contains", [])

# Étape 70 — PREUVE
print("[PREUVE étape 70] ValueSet CIM-11 expansé (code 04) :")
print_keys(cim11_expanded, "resourceType", "status")
print(f"Concepts retournés : {len(contains_cim11)}")
if contains_cim11:
    for c in contains_cim11[:5]:
        print(f"  {c.get('code')} — {c.get('display')}")

  → POST https://gazelle-proxypatans.kereval.cloud:10102/fhir/ValueSet/$expand
  ✓ runs/04_iti97_expand_20260311T211449/step070_expand-cim11-04.json
[PREUVE étape 70] ValueSet CIM-11 expansé (code 04) :
{
  "resourceType": "ValueSet"
}
Concepts retournés : 264
  4B40.1 — Abcès du thymus
  BlockL1-4A8 — Affections allergiques ou d'hypersensibilité
  4A8Y — Affections allergiques ou d'hypersensibilité d'un autre type précisé
  4A8Z — Affections allergiques ou d'hypersensibilité d'un autre type, sans précision
  4A85 — Affections complexes d'allergie ou d'hypersensibilité


---
## Récapitulatif

In [4]:
print(f"Run : {RUN_DIR}")
print(f"{'Fichier':<45} {'Taille':>10}")
print("-" * 57)
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, fname)
    size  = os.path.getsize(fpath)
    size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
    print(f"  {fname:<43} {size_str:>10}")
print(f"\n✓ {NOTEBOOK_ID} — terminé.")


Run : runs/04_iti97_expand_20260311T211449
Fichier                                           Taille
---------------------------------------------------------
  step030_expand-snomed-91723000.json             5.3 MB
  step070_expand-cim11-04.json                   47.7 KB

✓ 04_iti97_expand — terminé.
